In [4]:
import pandas as pd
import json
import os
import ast

In [2]:
def read_first_json_file(directory_path):
    for filename in os.listdir(directory_path):
        if filename.endswith('.json'):
            filepath = os.path.join(directory_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data, filename
    return None, None

data, filename = read_first_json_file('../json_file')
if data:
    df = pd.DataFrame(data)
    df = df.drop(columns=['description', 'statistics', 'channel_title'])
    print(f"Đã đọc file: {filename}")
    df.info()

Đã đọc file: truc_tiep_game_data.json
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   video_id      30 non-null     object
 1   title         30 non-null     object
 2   published_at  30 non-null     object
 3   comments      30 non-null     object
dtypes: object(4)
memory usage: 1.1+ KB


In [3]:

df_exploded = df.explode('comments', ignore_index=True)
df_comments = pd.json_normalize(df_exploded['comments'])
df_exploded['datetime'] = pd.to_datetime(df_exploded['published_at'])
df_exploded['date'] = df_exploded['datetime'].dt.date
df_exploded['time'] = df_exploded['datetime'].dt.time

cleaned_df_comments = pd.concat([df_exploded[['video_id']], df_comments], axis=1)
cleaned_df_comments = df_exploded.drop(columns=['published_at', 'datetime', 'title'])
cleaned_df_comments.to_csv('../data/comments.csv', index=False)

display(cleaned_df_comments.head(10))




,video_id,comments,date,time
0,Wmy5TQQPQho,"{'text': '❤', 'author': '@rockyjohnson2511', '...",2025-05-27,16:31:12
1,Wmy5TQQPQho,"{'text': 'Hài vc', 'author': '@sungchusinh', '...",2025-05-27,16:31:12
2,Wmy5TQQPQho,"{'text': 'game này hay nè, hóng hóng', 'author...",2025-05-27,16:31:12
3,Wmy5TQQPQho,"{'text': '❤', 'author': '@nguoianam', 'publish...",2025-05-27,16:31:12
4,Wmy5TQQPQho,{'text': 'Cuối cùng anh tôi cũng chơi game này...,2025-05-27,16:31:12
5,G5lS6O7j4RI,"{'text': 'In Game, vua của mọi nghề 😅😅', 'auth...",2025-05-30,17:48:58
6,G5lS6O7j4RI,"{'text': 'anh em mình cứ thế thôii hẹ hẹ hẹ', ...",2025-05-30,17:48:58
7,G5lS6O7j4RI,{'text': '28:35 còn thiếu phần bốc phét 100/10...,2025-05-30,17:48:58
8,G5lS6O7j4RI,"{'text': 'oh', 'author': '@leonad8416', 'publi...",2025-05-30,17:48:58
9,G5lS6O7j4RI,"{'text': 't', 'author': '@idolhuan9308', 'publ...",2025-05-30,17:48:58


In [12]:
df = pd.read_csv('../data/comments.csv')
def parse_comment_dict(comment_str):
    try:
        return ast.literal_eval(comment_str)
    except:
        return {}

df = df.drop(columns=['date', 'time'])
df['comment_dict'] = df['comments'].apply(parse_comment_dict)

df['comment_text'] = df['comment_dict'].apply(lambda x: x.get('text', ''))
df['author'] = df['comment_dict'].apply(lambda x: x.get('author', ''))
df['published_at'] = df['comment_dict'].apply(lambda x: x.get('published_at', ''))


df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
df['date'] = df['published_at'].dt.date
df['time'] = df['published_at'].dt.time

df = df.drop(columns=['published_at', 'comments', 'comment_dict'])

df.to_csv('../data/etc_comments.csv', index=False)
display(df.head(10))

,video_id,comment_text,author,date,time
0,Wmy5TQQPQho,❤,@rockyjohnson2511,2025-05-31,18:29:32
1,Wmy5TQQPQho,Hài vc,@sungchusinh,2025-05-31,16:43:21
2,Wmy5TQQPQho,"game này hay nè, hóng hóng",@dg.luceff2004,2025-05-31,12:10:06
3,Wmy5TQQPQho,❤,@nguoianam,2025-05-30,15:44:47
4,Wmy5TQQPQho,Cuối cùng anh tôi cũng chơi game này,@lehoangtuuc8713,2025-05-30,12:09:56
5,G5lS6O7j4RI,"In Game, vua của mọi nghề 😅😅",@thanhchelsea8753,2025-06-01,00:13:53
6,G5lS6O7j4RI,anh em mình cứ thế thôii hẹ hẹ hẹ,@hnahle-816,2025-05-31,20:31:45
7,G5lS6O7j4RI,28:35 còn thiếu phần bốc phét 100/100 nữa anh ...,@amegafuri,2025-05-31,18:42:22
8,G5lS6O7j4RI,oh,@leonad8416,2025-05-31,17:47:41
9,G5lS6O7j4RI,t,@idolhuan9308,2025-05-31,17:09:41
